# Memory Management

Memory allows agents to retain information across interactions. **Short-term memory** holds the current conversation context; **long-term memory** persists facts, preferences, and knowledge across sessions.

## Implementation with Flyte v2

This notebook reimplements the LangChain `ConversationBufferMemory` and LangGraph `InMemoryStore` patterns using **Flyte v2 + `MemoryStore`**.

#### LangChain / LangGraph vs Flyte v2 — Key Differences

| Aspect | LangChain / LangGraph | Flyte v2 |
|--------|----------------------|----------|
| **Short-term memory** | `ConversationBufferMemory` (in-process, lost on restart) | `store.messages` — managed transcript, auto-persisted to object storage |
| **Long-term memory** | LangGraph `InMemoryStore` (per-process, ephemeral) | `store.write_json("profile.json", ...)` — named artifact in the same Dir-backed store |
| **Persistence** | In-process only | `store.save()` → deterministic keyed path in Flyte object storage |
| **Observability** | Log parsing | Built-in audit log (`store.audit_tail()`) — every write logged with actor, sha256, timestamp |
| **Versioning** | None | Optional per-file version history (`keep_versions=True`) |
| **Secrets** | `.env` / `os.environ` | `flyte.Secret` injected by cluster |

### 1. Install dependencies

In [ ]:
!uv pip install 'flyte[tui]' anthropic 

### Start the devbox

If you haven't already, install the flyte package with the command above, then launch the local cluster:

In [ ]:
!flyte start devbox

### 2. Store your API key

In [ ]:
!flyte create secret ANTHROPIC_API_KEY --value sk-...

### 3. Import dependencies and configure the TaskEnvironment

In [5]:
import json
import os
from dataclasses import dataclass, field
from datetime import timedelta

from anthropic import AsyncAnthropic
import flyte
from flyte.ai.agents import MemoryStore

flyte.init_from_config()

_image = (
    flyte.Image.from_debian_base(name="memory-agent", python_version=(3, 12))
    .with_pip_packages("anthropic>=0.25.0")
)

memory_env = flyte.TaskEnvironment(
    name="memory_agent",
    image=_image,
    resources=flyte.Resources(cpu="1", memory="2Gi"),
    secrets=[
        flyte.Secret(key="ANTHROPIC_API_KEY", as_env_var="ANTHROPIC_API_KEY"),
    ],
)

### 4. Define the data models

`ConversationMemory` and `UserProfile` are local helper dataclasses used **inside** the task — they never appear in the Flyte type signature, so there are no serializer constraints on their field types.

`MemoryStore` handles persistence: the transcript lives in `store.messages`, and `UserProfile` is written as `profile.json` via `write_json()`. Both survive pod restarts because the store is saved to a deterministic keyed path in Flyte's object storage after every turn.

```
<storage_root>/agents/memory-store/v0/{org}/{project}/{domain}/{session_key}/
    messages.json       ← transcript (store.messages)
    profile.json        ← UserProfile artifact
    audit/log.jsonl     ← every write: actor, sha256, timestamp, reason
```
[Learn more about Flyte's MemoryStore](https://www.union.ai/docs/v2/union/api-reference/flyte-sdk/packages/flyte.ai.agents/memorystore/)

In [6]:
@dataclass
class ConversationMemory:
    """Short-term: current conversation thread. Replaces LangChain ConversationBufferMemory."""
    messages: list[dict[str, str]] = field(default_factory=list)

    def append(self, role: str, content: str) -> "ConversationMemory":
        return ConversationMemory(messages=self.messages + [{"role": role, "content": content}])

    def to_api_format(self) -> list[dict]:
        return list(self.messages)


@dataclass
class UserProfile:
    """Long-term: user facts that persist across sessions. Replaces LangGraph InMemoryStore."""
    name: str = ""
    preferences: dict[str, str] = field(default_factory=dict)
    visited_places: list[str] = field(default_factory=list)

    def remember(self, key: str, value: str) -> "UserProfile":
        return UserProfile(
            name=self.name,
            preferences={**self.preferences, key: value},
            visited_places=self.visited_places,
        )

    def add_visit(self, place: str) -> "UserProfile":
        return UserProfile(
            name=self.name,
            preferences=self.preferences,
            visited_places=self.visited_places + [place],
        )

    def summary(self) -> str:
        if not self.preferences and not self.visited_places:
            return "No user profile information available."
        parts = []
        if self.name:
            parts.append(f"User: {self.name}")
        if self.preferences:
            parts.append("Preferences: " + ", ".join(f"{k}={v}" for k, v in self.preferences.items()))
        if self.visited_places:
            parts.append("Previously visited: " + ", ".join(self.visited_places))
        return ". ".join(parts)

### 5. Define the travel agent task

`chat_turn` takes just two inputs — `user_message` and `session_key` — and returns the agent's response as a plain `str`. All memory state lives inside `MemoryStore`:

1. `MemoryStore.get_or_create.aio(key=session_key)` — loads the existing store on turns 2+, creates it on turn 1.
2. `store.read_json("profile.json")` — rehydrates `UserProfile` from the stored artifact.
3. `store.append(...)` — appends user and assistant messages to the managed transcript.
4. `store.write_json("profile.json", ...)` — persists updated profile facts as a named artifact.
5. `store.save.aio()` — flushes transcript + artifacts to the deterministic keyed path in object storage.

`@flyte.trace` on `_chat` and `_extract_facts` checkpoints each LLM call, so a pod failure mid-turn retries from the last checkpoint rather than replaying the whole turn.

**How is this different from LangChain:** `ConversationBufferMemory` is stateful in-process — gone on any restart. `MemoryStore` keys memory to a session ID in Flyte's object storage. Any task, on any pod, at any time, can load the exact same store with just the key.

In [7]:
TRAVEL_AGENT_SYSTEM = """\
You are a friendly travel agent with excellent memory. Help the user plan trips.
When the user mentions a preference (budget, travel style, food, etc.), acknowledge it.
When they mention a place they've visited, acknowledge it.
Keep responses concise (2-3 sentences max)."""

PROFILE_EXTRACTOR_SYSTEM = """\
Extract structured facts from this conversation turn.
Respond ONLY with JSON in this exact format:
{"name": "<user name if mentioned, else empty string>",
 "preference_key": "<one preference key if any, else empty string>",
 "preference_value": "<preference value, else empty string>",
 "visited_place": "<place name if user mentioned visiting, else empty string>"}"""


@flyte.trace
async def _chat(messages: list[dict], system: str) -> str:
    client = AsyncAnthropic()
    response = await client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=512,
        system=system,
        messages=messages,
    )
    return response.content[0].text


@flyte.trace
async def _extract_facts(user_message: str, agent_response: str) -> dict:
    client = AsyncAnthropic()
    response = await client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=256,
        system=PROFILE_EXTRACTOR_SYSTEM,
        messages=[{
            "role": "user",
            "content": f"User said: {user_message}\nAgent replied: {agent_response}",
        }],
    )
    raw = response.content[0].text.strip()
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
    try:
        return json.loads(raw)
    except Exception:
        return {}


@memory_env.task(
    retries=2,
    timeout=timedelta(minutes=5),
    cache=flyte.Cache(behavior="disable"),
)
async def chat_turn(user_message: str, session_key: str) -> str:
    """One conversation turn with MemoryStore-backed memory."""
    store = await MemoryStore.get_or_create.aio(key=session_key)

    profile_data = await store.read_json.aio("profile.json", default={})
    profile = UserProfile(
        name=profile_data.get("name", ""),
        preferences=profile_data.get("preferences", {}),
        visited_places=profile_data.get("visited_places", []),
    )

    profile_context = profile.summary()
    system = TRAVEL_AGENT_SYSTEM
    if profile_context != "No user profile information available.":
        system += f"\n\nKnown user context: {profile_context}"

    store.append({"role": "user", "content": user_message})
    response = await _chat(messages=store.messages, system=system)
    store.append({"role": "assistant", "content": response})

    facts = await _extract_facts(user_message=user_message, agent_response=response)
    if facts.get("name"):
        profile = UserProfile(name=facts["name"], preferences=profile.preferences, visited_places=profile.visited_places)
    if facts.get("preference_key") and facts.get("preference_value"):
        profile = profile.remember(facts["preference_key"], facts["preference_value"])
    if facts.get("visited_place"):
        profile = profile.add_visit(facts["visited_place"])

    await store.write_json.aio(
        "profile.json",
        {"name": profile.name, "preferences": profile.preferences, "visited_places": profile.visited_places},
        actor="chat_turn",
        reason=user_message[:60],
    )
    await store.save.aio()
    return response

### 6. Run a multi-turn conversation

Each `flyte.run.aio()` call dispatches a single turn. The run loop no longer tracks any memory state — `session_key` is the only thread between turns. `MemoryStore.get_or_create` inside `chat_turn` does the rest.

In [ ]:
CONVERSATION = [
    "Hi! I'm planning a trip and could use some help.",
    "I love budget travel and prefer hostels over hotels.",
    "I visited Tokyo last year and it was amazing.",
    "What would you recommend for Southeast Asia on a tight budget?",
    "I'm a vegetarian, so food options matter a lot to me.",
]
SESSION_KEY = "travel-agent-david"

for i, user_msg in enumerate(CONVERSATION, 1):
    print(f"─── Turn {i} " + "─" * 50)
    print(f"User: {user_msg}")

    run = await flyte.run.aio(chat_turn, user_message=user_msg, session_key=SESSION_KEY)
    response: str = run.outputs()[0]

    print(f"Agent: {response}")
    print(f"Run:   {run.url}")
    print()

In [ ]:
# Inspect the final memory state by loading the store from inside a task.
# MemoryStore.get_or_create requires a Flyte raw-data context, so we read it
# via a thin wrapper task rather than directly from the notebook.

@memory_env.task(cache=flyte.Cache(behavior="disable"))
async def read_memory(session_key: str) -> str:
    import json as _json
    store = await MemoryStore.get_or_create.aio(key=session_key)
    profile = await store.read_json.aio("profile.json", default={})
    audit = await store.audit_tail(20)
    return _json.dumps({"turns": len(store.messages) // 2, "profile": profile, "audit": audit}, indent=2)

run = await flyte.run.aio(read_memory, session_key=SESSION_KEY)
import json
state = json.loads(run.outputs()[0])

print(f"Conversation turns stored: {state['turns']}")
print(f"Profile: {state['profile']}")
print(f"\nAudit trail ({len(state['audit'])} events):")
for event in state["audit"]:
    print(f"  {event['ts']}  [{event['op']:6}]  {event['path']}  — {event.get('reason', '')[:50]}")

## Scaling the pattern

For high-throughput conversational systems, the `ReusePolicy` eliminates cold-start cost per turn. Each pod handles multiple concurrent conversations — `session_key` is the only per-request input, and `MemoryStore.get_or_create` loads the correct store from object storage on every call. There is no shared mutable state between requests.

> **Note:** `ReusePolicy` is a Union-specific feature that requires a [Union deployment](https://www.union.ai/docs/v2/union/). It is not supported on the local devbox.

In [ ]:
# Production: handle many concurrent chat sessions
production_memory_env = flyte.TaskEnvironment(
    name="memory_agent_prod",
    image=_image,
    resources=flyte.Resources(cpu="2", memory="4Gi"),
    secrets=[flyte.Secret(key="ANTHROPIC_API_KEY", as_env_var="ANTHROPIC_API_KEY")],
    reusable=flyte.ReusePolicy(
        replicas=(2, 8),
        concurrency=16,
        scaledown_ttl=timedelta(minutes=5),
        idle_ttl=timedelta(minutes=15),
    ),
)